# Day 14: Supervisor Pattern — One Agent Delegates to Specialists

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> A good manager doesn't do everything. They delegate to the right specialist.

The supervisor pattern is the production version of routing. A "manager" agent
receives all requests and delegates to specialist workers. This is the pattern behind
most real-world agent systems.

## What You'll Build
- A supervisor agent with 3 specialist workers
- The supervisor reads the request and picks the right worker
- Same pattern in all 3 frameworks — this is Week 2's culmination

In [1]:
# ── Setup ───────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 14 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (10 model(s) available)

DAY 14 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


## Today's Architecture

```
                    ┌──────────────┐
   User Input  ──>  │  Supervisor  │
                    └──────┬───────┘
                           │
              ┌────────────┼────────────┐
              │            │            │
     ┌────────▼──┐  ┌──────▼──────┐  ┌──▼───────────┐
     │    Math    │  │  Geography  │  │   General    │
     │  Specialist │  │  Specialist │  │  Specialist  │
     └────────────┘  └─────────────┘  └──────────────┘
```

Test questions:
1. "What is 15% of 2340?" → Math
2. "What is the capital of France?" → Geography
3. "Explain machine learning in 2 sentences." → General

---
## Framework 1: OpenAI Agents SDK — Handoffs

In [6]:
from agents import Agent, Runner, function_tool

model = get_openai_agents_model()

# ── Workers ────────────────────────────────────────────
@function_tool
def calculate(expr: str) -> str:
    """Evaluate a math expression."""
    import re
    p = re.match(r"(\d+(?:\.\d+)?)\s*%\s*of\s*(\d+(?:\.\d+)?)", expr)
    if p: return str(float(p.group(2)) * float(p.group(1)) / 100)
    try: return str(eval(expr))
    except: return f"Cannot evaluate: {expr}"

# FIX: agent names must be valid tool-name tokens (no spaces).
# The SDK auto-generates a `transfer_to_<name>` handoff tool per agent; a space in
# the name triggered "contains invalid characters... transformed" warnings on every call.
math_worker = Agent(name="MathWorker", instructions="Solve math problems using the calculator tool.", tools=[calculate], model=model)
geo_worker = Agent(name="GeoWorker", instructions="Answer geography questions concisely.", model=model)
general_worker = Agent(name="GeneralWorker", instructions="Answer general knowledge questions.", model=model)

# ── Supervisor ────────────────────────────────────────
supervisor = Agent(
    name="Supervisor",
    instructions=(
        "You are a supervisor. Delegate to the right worker:\n"
        "- Math/calculations → MathWorker\n"
        "- Geography/capitals/countries → GeoWorker\n"
        "- Everything else → GeneralWorker\n"
        "Hand off immediately. Do not answer yourself."
    ),
    handoffs=[math_worker, geo_worker, general_worker],
    model=model,
)

tests = ["What is 15% of 2340?", "What is the capital of France?", "Explain machine learning in 2 sentences."]

print("=" * 60)
print("OPENAI AGENTS SDK — SUPERVISOR PATTERN")
print("=" * 60)

for q in tests:
    result = await Runner.run(supervisor, q)
    # result.last_agent is the agent that produced the final output — i.e. the worker
    # the supervisor handed off to. Matches the "Routed to:" line in the LangGraph cell.
    print(f"\nQ: {q}")
    print(f"Routed to: {result.last_agent.name}")
    print(f"A: {result.final_output[:200]}")

OPENAI AGENTS SDK — SUPERVISOR PATTERN

Q: What is 15% of 2340?
Routed to: MathWorker
A: 15% of 2340 is 351.0.

Q: What is the capital of France?
Routed to: GeoWorker
A: The capital of France is Paris.

Q: Explain machine learning in 2 sentences.
Routed to: Supervisor
A: Machine learning is a subset of artificial intelligence that focuses on the development of algorithms and models that enable computers to learn from and make predictions or decisions based on data, wi


---
## Framework 2: LangGraph — Supervisor Node

In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from typing_extensions import TypedDict, Annotated
import operator

model = get_langgraph_model()

class SupervisorState(TypedDict):
    messages: Annotated[list, operator.add]
    department: str

def supervisor_node(state: SupervisorState) -> dict:
    user_msg = state["messages"][-1].content
    resp = model.invoke([
        SystemMessage(content="Classify: 'math', 'geography', or 'general'. Return ONLY the word."),
        HumanMessage(content=user_msg),
    ])
    dept = resp.content.strip().lower()
    for d in ["math", "geography", "general"]:
        if d in dept: dept = d; break
    else: dept = "general"
    return {"department": dept}

def math_node(state: SupervisorState) -> dict:
    r = model.invoke([SystemMessage(content="Solve math problems step by step."), state["messages"][-1]])
    return {"messages": [r]}

def geo_node(state: SupervisorState) -> dict:
    r = model.invoke([SystemMessage(content="Answer geography questions concisely."), state["messages"][-1]])
    return {"messages": [r]}

def general_node(state: SupervisorState) -> dict:
    r = model.invoke([SystemMessage(content="Answer general knowledge questions."), state["messages"][-1]])
    return {"messages": [r]}

def route(state: SupervisorState) -> str:
    return state.get("department", "general")

builder = StateGraph(SupervisorState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("math", math_node)
builder.add_node("geography", geo_node)
builder.add_node("general", general_node)
builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", route, {"math": "math", "geography": "geography", "general": "general"})
builder.add_edge("math", END)
builder.add_edge("geography", END)
builder.add_edge("general", END)

graph = builder.compile()

tests = ["What is 15% of 2340?", "What is the capital of France?", "Explain machine learning in 2 sentences."]

print("=" * 60)
print("LANGGRAPH — SUPERVISOR PATTERN")
print("=" * 60)

for q in tests:
    result = graph.invoke({"messages": [HumanMessage(content=q)], "department": ""})
    print(f"\nQ: {q}")
    print(f"Routed to: {result['department']}")
    print(f"A: {result['messages'][-1].content[:200]}")

LANGGRAPH — SUPERVISOR PATTERN

Q: What is 15% of 2340?
Routed to: math
A: To find 15% of 2340, we can use the following steps:

Step 1: Understand that "percent" means "per hundred". So, 15% can be written as a decimal. To do this, divide 15 by 100.
\[ 15\% = \frac{15}{100}

Q: What is the capital of France?
Routed to: general
A: The capital of France is Paris.

Q: Explain machine learning in 2 sentences.
Routed to: general
A: Machine learning is a subset of artificial intelligence that allows software applications to grow and improve their performance from the data they are given without human intervention, by using algori


---
## Framework 3: CrewAI — Hierarchical Process

In [4]:
from crewai import Agent, Task, Crew, Process

llm = get_crewai_llm()

# ── Workers ────────────────────────────────────────────
math_w = Agent(role="Math Specialist", goal="Solve math", backstory="Math expert.", llm=llm, verbose=True)
geo_w = Agent(role="Geography Specialist", goal="Answer geo questions", backstory="Geography expert.", llm=llm, verbose=True)
general_w = Agent(role="General Specialist", goal="Answer general questions", backstory="Polymath.", llm=llm, verbose=True)

tests = [
    ("What is 15% of 2340?", math_w),
    ("What is the capital of France?", geo_w),
    ("Explain machine learning in 2 sentences.", general_w),
]

print("=" * 60)
print("CREWAI — SUPERVISOR PATTERN")
print("=" * 60)

for q, specialist in tests:
    task = Task(description=q, expected_output="Your answer.", agent=specialist)
    crew = Crew(agents=[specialist], tasks=[task], process=Process.sequential, verbose=False)
    # FIX: kickoff_async() returns a coroutine — must be awaited.
    # Without await, str(result) printed "<coroutine object Crew.kickoff_async at 0x...>".
    # Sync kickoff() won't work either: Jupyter's event loop is already running, so
    # CrewAI refuses with "invoked synchronously from within a running event loop".
    result = await crew.kickoff_async()
    print(f"\nQ: {q}")
    print(f"Worker: {specialist.role}")
    print(f"A: {str(result)[:200]}")

CREWAI — SUPERVISOR PATTERN


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Specialist                                                                                         │
│                                                                                                                 │
│  Task: What is 15% of 2340?                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Specialist                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  To find 15% of 2340, we first convert the percentage to its decimal form.                                      │
│                                                                                                                 │
│  In this case, 15% is equivalent to \( \frac{15}{100} = 0.15 \).                                                │
│                                                                                                                 │
│  Next, we multiply 2340 by 0.15:                                                                                │
│                                                                                                                 │
│  \[                                                                                                             │
│  2340 \times 0.15 = 351                                                                                         │
│  \]                                                                                                             │
│                                                                                                                 │
│  Thus, the answer is 351.                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Q: What is 15% of 2340?
Worker: Math Specialist
A: To find 15% of 2340, we first convert the percentage to its decimal form. 

In this case, 15% is equivalent to \( \frac{15}{100} = 0.15 \).

Next, we multiply 2340 by 0.15:

\[
2340 \times 0.15 = 351



...


Q: What is the capital of France?
Worker: Geography Specialist
A: The capital of France is Paris.


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: General Specialist                                                                                      │
│                                                                                                                 │
│  Task: Explain machine learning in 2 sentences.                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: General Specialist                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Machine learning is a subset of artificial intelligence that enables systems to improve their performance and  │
│  accuracy on a specific task over time through algorithms training on large datasets. Essentially, it allows    │
│  computers to learn patterns without being explicitly programmed, making predictions or decisions by analyzing  │
│  data inputs.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Q: Explain machine learning in 2 sentences.
Worker: General Specialist
A: Machine learning is a subset of artificial intelligence that enables systems to improve their performance and accuracy on a specific task over time through algorithms training on large datasets. Essen


---
## Comparison: Supervisor Pattern

| Framework | Supervisor mechanism | Delegation is... | Complexity |
|---|---|---|---|
| OpenAI SDK | `handoffs=[workers]` | LLM-decided | Low |
| LangGraph | Conditional edges + supervisor node | Code-decided | Medium |
| CrewAI | `Process.hierarchical` | Manager-decided | Low-Medium |

**Key insight:** The supervisor pattern is the bridge between single-agent and
multi-agent systems. OpenAI SDK's handoffs are the most elegant. LangGraph gives
the most control over the delegation logic. CrewAI's hierarchical process adds
a manager LLM that orchestrates workers automatically.

## Week 2 Complete!

This week we added **intelligence** to our agents:
- Day 8: Conditional routing
- Day 9: Loops & iteration
- Day 10: Human-in-the-loop
- Day 11: State persistence
- Day 12: Guardrails
- Day 13: Evaluation
- Day 14: Supervisor pattern

**Week 3 preview:** Multi-agent teams, framework deep dives, and the ultimate
side-by-side comparison.

---
